### Joins, Relational Model, and Window Functions
Objectives :
- Strengthen your understanding of multi-table joins;
- Understand cardinality;
- Review CTEs, subqueries, and window functions.

In [0]:
from pyspark.sql import functions as F

#### 2.1 - Joins on client and transaction tables 

In [0]:
df_customers = spark.read.table('samples.bakehouse.sales_customers')
df_transactions = spark.read.table('samples.bakehouse.sales_transactions')

In [0]:
df_customers.printSchema()

In [0]:
df_transactions.printSchema()

##### Join sales data with customer data

In [0]:
%sql 
SELECT transactionID, dateTime, product, quantity, totalPrice, first_name, last_name
FROM samples.bakehouse.sales_transactions t
JOIN samples.bakehouse.sales_customers c ON t.customerID = c.customerID
ORDER BY t.dateTime DESC
LIMIT 10

In [0]:
df_transactions.join(df_customers, on='customerID', how='left').show(5)

##### Identify customers with no purchases using LEFT ANTI JOIN

In [0]:
%sql 
-- transaction table have all the customers 
SELECT transactionID, dateTime, product, quantity, totalPrice
FROM samples.bakehouse.sales_transactions t
LEFT ANTI JOIN samples.bakehouse.sales_customers c ON t.customerID = c.customerID
ORDER BY t.dateTime DESC

In [0]:
df_transactions.join(df_customers, on='customerID', how='left_anti').show(10)

##### Identify customers who have made purchases using LEFT SEMI JOIN

In [0]:
%sql 
SELECT transactionID, dateTime, product, quantity, totalPrice
FROM samples.bakehouse.sales_transactions t
LEFT SEMI JOIN samples.bakehouse.sales_customers c ON t.customerID = c.customerID
ORDER BY t.dateTime DESC
LIMIT 10

In [0]:
df_transactions.join(df_customers, on='customerID', how='left_semi').show(10)

##### Calculate revenue and the number of transactions by country/continent

In [0]:
%sql 
SELECT SUM(totalPrice) as totalPirce, COUNT(transactionID) as transactionCount, country 
FROM samples.bakehouse.sales_transactions 
JOIN samples.bakehouse.sales_customers USING (customerID)
GROUP BY country

In [0]:
df_join = df_transactions.join(df_customers, on='customerID', how='left')
df_join.groupBy('country').agg(
    F.sum('totalPrice').alias('totalPrice'),
    F.count('transactionID').alias('transactionCount')
).orderBy('totalPrice', ascending=False).show()

##### Calculate each region’s share of total revenue

In [0]:
%sql

SELECT SUM(totalPrice), state 
FROM samples.bakehouse.sales_transactions 
JOIN samples.bakehouse.sales_customers USING (customerID)
GROUP BY state

In [0]:
df_join.groupBy('state').agg(
    F.sum('totalPrice').alias('totalPrice')
).show()

#### 2.2 - Franchises and suppliers tables

In [0]:
df_franchises = spark.read.table("samples.bakehouse.sales_franchises")
df_suppliers = spark.read.table("samples.bakehouse.sales_suppliers")

In [0]:
df_franchises.printSchema()
df_suppliers.printSchema()

##### Calculate revenue and sales volume by franchise


In [0]:
%sql 

SELECT f.name as franchiseName, SUM(totalPrice) as total_price, SUM(quantity) as total_quantity
FROM samples.bakehouse.sales_transactions t
JOIN samples.bakehouse.sales_franchises f USING (franchiseID)
GROUP BY f.name
ORDER BY total_price DESC 

In [0]:
df_franchises_transactions = df_transactions.join(df_franchises, on='franchiseID', how='inner')

df_franchises_transactions.groupBy('name').agg(
    F.sum('totalPrice').alias('totalPrice'),
    F.sum('quantity').alias('quantity')
).orderBy('totalPrice', ascending=False).show()


##### Count unique customers by franchise


In [0]:
%sql 
SELECT COUNT(DISTINCT t.customerID) as customerCountUnique, f.name as franchiseName
FROM samples.bakehouse.sales_transactions t
JOIN samples.bakehouse.sales_franchises f USING (franchiseID)
GROUP BY f.name
ORDER BY customerCountUnique DESC

In [0]:
df_franchises_transactions.groupBy('name').agg(
    F.count('customerID').alias('customerCountUnique')
).orderBy('customerCountUnique', ascending=False).show()

##### Calculate the revenue associated with each supplier through its franchise

In [0]:
%sql 
SELECT s.name as supplierName, SUM(totalPrice) as totalPrice
FROM samples.bakehouse.sales_transactions t
LEFT JOIN samples.bakehouse.sales_franchises f USING (franchiseID)
LEFT JOIN samples.bakehouse.sales_suppliers s USING(supplierID)
GROUP BY s.name
ORDER BY totalPrice DESC


In [0]:
df_result = (
    df_transactions
    .join(df_franchises, on="franchiseID", how="left")
    .join(df_suppliers, on="supplierID", how="left")
    .select(
        df_transactions["*"],
        df_franchises["name"].alias("franchiseName"),
        df_suppliers["name"].alias("supplierName")
    )
    )

df_result.groupBy('supplierName').agg(
    F.sum('totalPrice').alias('totalPrice')
    ).orderBy('totalPrice', ascending=False).show()



There are NULL values in supplierName

In [0]:
# exam the NULL value in supplierName :find the supplierID present in franchise table but not in supplier table. 
(
    df_transactions
    .join(df_franchises, on="franchiseID", how="left")
    .join(df_suppliers, on="supplierID", how="left_anti")
    .select("supplierID")
    .distinct()
    .show()
)

So supplier table doesn't have those supplierID

#### 2.3 - Window Functions 

In [0]:
from pyspark.sql import Window

##### Top 3 products by day

In [0]:
%sql 
WITH product_daily_revenue AS(
    SELECT product, TO_DATE(datetime) AS transa_day, SUM(totalPrice) AS daily_revenue
    FROM samples.bakehouse.sales_transactions 
    GROUP BY product, TO_DATE(datetime)
    ORDER BY transa_day, daily_revenue DESC),
    ranked_products AS(
    SELECT *, RANK() OVER(PARTITION BY transa_day ORDER BY daily_revenue DESC) AS rank
    FROM product_daily_revenue)

SELECT *
FROM ranked_products
WHERE rank <= 3
ORDER BY transa_day, rank


In [0]:
df_product_daily = (
    df_transactions
    .withColumn('transa_date', F.col('dateTime').cast('date'))
    .groupBy('product', 'transa_date')
    .agg(F.sum('totalPrice').alias('daily_revenue'))
    .orderBy('transa_date', 'daily_revenue', ascending=False))

df_ranked_product = (
    df_product_daily
    .withColumn(
        'rank', 
        F.dense_rank()
        .over(
            Window.partitionBy('transa_date')
            .orderBy(F.desc('daily_revenue'))
            )
    )
)

df_ranked_product.filter('rank <= 3').display()

##### Top 3 franchises by country or state 

In [0]:
%sql 
WITH franchise_country_revenue AS(
    SELECT f.name as franchise, country, SUM(t.totalPrice) AS franchise_revenue
    FROM samples.bakehouse.sales_transactions t 
    LEFT JOIN samples.bakehouse.sales_franchises f USING (franchiseID)
    GROUP BY f.name, country),
    ranked_franchise_revenue AS(
        SELECT *, DENSE_RANK() OVER(PARTITION BY COUNTRY ORDER BY franchise_revenue DESC) AS rank
        FROM franchise_country_revenue)
    
SELECT *
FROM ranked_franchise_revenue
WHERE rank <= 3 
ORDER BY country, rank

In [0]:
df_franchises_country_revenue = (
    df_transactions
    .join(df_franchises, on='franchiseID', how='left')
    .groupBy(F.col('name').alias('franchise'), 'country')
    .agg(F.sum('totalPrice').alias('franchise_revenue'))
    .orderBy(F.desc('franchise_revenue'))
)

df_ranked_franchises_revenue = (
    df_franchises_country_revenue
    .withColumn(
        'rank',
        F.dense_rank()
        .over(
            Window.partitionBy('country')
            .orderBy(F.desc('franchise_revenue'))
        )
    )
)

df_ranked_franchises_revenue.filter('rank <= 3').display()

##### Calculate cumulative daily / weekly revenue

In [0]:
%sql
WITH daily_cumul_revenue AS (
    SELECT TO_DATE(datetime) as transa_day, SUM(totalPrice) as daily_revenue
    FROM samples.bakehouse.sales_transactions
    GROUP BY TO_DATE(datetime)
    ORDER BY transa_day)

SELECT *, SUM(daily_revenue) OVER(ORDER BY transa_day) as cumul_revenue
FROM daily_cumul_revenue

In [0]:
%sql
WITH weekly_cumul__revenue AS (
    SELECT WEEKOFYEAR(datetime) as transa_week, SUM(totalPrice) as weekly_revenue
    FROM samples.bakehouse.sales_transactions
    GROUP BY WEEKOFYEAR(datetime)
)

SELECT *, SUM(weekly_revenue) OVER(ORDER BY transa_week) as cumul_revenue
FROM weekly_cumul__revenue

In [0]:
df_daily_revenue = (
    df_transactions
    .groupBy(F.col('dateTime').cast('date').alias('transa_day'))
    .agg(F.sum('totalPrice').alias('daily_revenue'))
)

df_daily_revenue.withColumn('cumul_daily_revenue', F.sum('daily_revenue').over(Window.orderBy('transa_day'))).display()

##### Identify each customer’s first and last transaction

In [0]:
%sql
SELECT customerID, DATE(MIN(dateTime)) as first_transa, DATE(MAX(dateTime)) AS last_transa
FROM samples.bakehouse.sales_transactions
GROUP BY customerID
LIMIT 10 

In [0]:
(
    df_transactions
    .groupBy('customerID')
    .agg(
        F.min('dateTime').cast('date').alias('first_transa'),
        F.max('dateTime').cast('date').alias('last_transa')
    )
).display()

##### Calculate the time between two consecutive purchases for each customer

In [0]:
%sql 
WITH purchase AS (
    SELECT customerID, DATE(dateTime) AS transa_date, LAG(DATE(dateTime)) OVER(PARTITION BY customerID ORDER BY DATE(dateTime)) AS prev_transa_date
    FROM samples.bakehouse.sales_transactions
)

SELECT *, DATE_DIFF(transa_date, prev_transa_date) AS gap
FROM purchase
WHERE prev_transa_date IS NOT NULL

In [0]:
(
    df_transactions
    .select(
        F.col('customerID'),
        F.col('dateTime').cast('date').alias('transa_date'))
    .withColumn(
        'prev_transa_date', 
        F.lag('transa_date').over(
            Window.partitionBy('customerID')
            .orderBy('transa_date'))
    )
    .withColumn(
        'days_between', 
        F.datediff('transa_date', 'prev_transa_date')
    )
).filter(F.col('prev_transa_date').isNotNull()).display()
    
    
